# Reverse Convertible Pricing — From Scratch

This notebook prices a Reverse Convertible (RC) and a Multi-Barrier Reverse Convertible (MBRC) using options theory.

## Key intuition

A reverse convertiblefair market value driven by:
- Time to maturity
- Implied volatility of the underlying(s)
- Interest rates
- Correlation (for worst-of products)

### Decomposition
```
RC = Zero-Coupon Bond  +  Coupon PV  -  Short Put
```

The investor **sells a put** to the issuer and receives the coupon as premium. If the stock falls below the barrier at maturity, the investor gets shares instead of cash — exactly like a put being exercised against them.

For a **worst-of** product (MBRC), the investor sells a **worst-of put**: the put is exercised on whichever stock performed worst. This is more valuable to the issuer (higher premium → higher coupon) because worst-of options are worth more than single-name puts.

In [1]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

---
## Part 1 — Black-Scholes building blocks

In [ ]:
def bs_put(S, K, T, r, sigma):
    """
    Black-Scholes European put price.

    Parameters
    ----------
    S     : current spot price
    K     : strike price
    T     : time to maturity in years
    r     : risk-free rate (continuous, annualised)
    sigma : implied volatility (annualised)
    """
    if T <= 0:
        return max(K - S, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


# Quick sanity check
p = bs_put(S=100, K=100, T=1, r=0.03, sigma=0.20)
print(f"ATM put (S=100, K=100, T=1y, r=3%, vol=20%) = {p:.4f}")

---
## Part 2 — Single-underlying BRC pricer

In [ ]:
class BRC:
    """
    Barrier Reverse Convertible — single underlying, European barrier.

    Structure
    ---------
    - Investor pays `notional` today
    - At maturity receives: coupon  +  max(notional, notional * S_T/K)
      - If S_T >= K (barrier = strike): full notional back
      - If S_T <  K: notional * (S_T / K)  — loss proportional to downside

    Equivalent to
    -------------
    Bond (PV of notional)  +  PV of coupon  -  Put(S, K, T, r, sigma)
    All quantities per unit of notional.
    """

    def __init__(self, S, K, T, r, sigma, coupon_rate, notional=1.0):
        """
        S            : current spot
        K            : strike (= barrier level in absolute terms)
        T            : years to maturity
        r            : risk-free rate (annual, continuous)
        sigma        : implied vol of underlying
        coupon_rate  : annual coupon rate (e.g. 0.08 for 8%)
        notional     : face value (default 1.0 for % terms)
        """
        self.S = S
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.coupon_rate = coupon_rate
        self.notional = notional

    def bond_pv(self):
        """PV of receiving notional at maturity."""
        return self.notional * np.exp(-self.r * self.T)

    def coupon_pv(self):
        """PV of total coupon (paid at maturity, lump sum)."""
        total_coupon = self.notional * self.coupon_rate * self.T
        return total_coupon * np.exp(-self.r * self.T)

    def put_value(self):
        """Value of the put the investor is short."""
        # Put is on (notional/K) shares, struck at K
        # = notional * bs_put(S/K normalised to 1)
        # More simply: bs_put in absolute terms
        return (self.notional / self.K) * bs_put(self.S, self.K, self.T, self.r, self.sigma)

    def fair_value(self):
        """RC fair value = bond + coupon - short put."""
        return self.bond_pv() + self.coupon_pv() - self.put_value()

    def fair_value_pct(self):
        """Fair value as % of notional."""
        return self.fair_value() / self.notional

    # ---- Numerical Greeks (bump-and-reprice) ----

    def _reprice(self, **kwargs):
        params = dict(S=self.S, K=self.K, T=self.T, r=self.r,
                      sigma=self.sigma, coupon_rate=self.coupon_rate,
                      notional=self.notional)
        params.update(kwargs)
        return BRC(**params).fair_value()

    def delta(self, dS=0.01):
        """dV/dS — how much the RC value changes per 1 unit move in spot."""
        return (self._reprice(S=self.S + dS) - self._reprice(S=self.S - dS)) / (2 * dS)

    def gamma(self, dS=0.01):
        """d²V/dS² — convexity of RC value with respect to spot."""
        return (self._reprice(S=self.S + dS) - 2*self.fair_value() + self._reprice(S=self.S - dS)) / (dS**2)

    def vega(self, dvol=0.001):
        """dV/d(sigma) per 1% vol move (divide result by 100 for 1bp)."""
        return (self._reprice(sigma=self.sigma + dvol) - self._reprice(sigma=self.sigma - dvol)) / (2 * dvol)

    def theta(self, dt=1/365):
        """dV/dt — value decay per calendar day (T decreasing)."""
        if self.T - dt <= 0:
            return 0.0
        return (self._reprice(T=self.T - dt) - self.fair_value()) / dt

In [ ]:
# Example: Novartis BRC
# Spot=123, Strike=123 (at-the-money), 6 months left, coupon=8% p.a., vol=20%

brc = BRC(
    S=123.0,       # current spot
    K=123.0,       # strike = barrier
    T=0.5,         # 6 months
    r=0.03,        # 3% risk-free
    sigma=0.20,    # 20% implied vol
    coupon_rate=0.08,
    notional=100_000
)

print(f"Bond PV     : {brc.bond_pv():>12,.2f}")
print(f"Coupon PV   : {brc.coupon_pv():>12,.2f}")
print(f"Put value   : {brc.put_value():>12,.2f}   ← investor is SHORT this")
print(f"────────────────────────────────")
print(f"Fair value  : {brc.fair_value():>12,.2f}")
print(f"Fair value% : {brc.fair_value_pct():>12.4f}")
print()
print(f"Delta       : {brc.delta():>12.4f}   (per 1 CHF spot move)")
print(f"Gamma       : {brc.gamma():>12.6f}")
print(f"Vega        : {brc.vega():>12.2f}   (per 10% vol move = x10)")
print(f"Theta       : {brc.theta():>12.2f}   (per calendar day)")

---
## Part 3 — Worst-of MBRC via Monte Carlo

For a multi-underlying RC, the put is on the **worst performer**. Closed-form solutions require copulas. Monte Carlo is the cleanest approach for learning.

We simulate correlated GBM paths for all underlyings, compute worst-of performance at maturity, then average the put payoffs.

In [ ]:
class MBRC:
    """
    Multi-asset Barrier Reverse Convertible — worst-of, European barrier.

    Priced via Monte Carlo simulation of correlated GBM.

    Fair value = Bond PV  +  Coupon PV  -  Worst-Of Put (MC)

    The worst-of put pays: max(K_wof - S_wof_T, 0) where S_wof_T is the
    worst performing underlying at maturity (normalised to 1.0 at inception).
    """

    def __init__(self, spots, strikes, T, r, sigmas, corr_matrix,
                 coupon_rate, notional=1.0, n_sims=100_000, seed=42):
        """
        spots        : list of current spot prices  [S1, S2, ...]
        strikes      : list of strike prices        [K1, K2, ...]
        T            : years to maturity
        r            : risk-free rate
        sigmas       : list of implied vols          [v1, v2, ...]
        corr_matrix  : correlation matrix (n x n numpy array)
        coupon_rate  : annual coupon
        notional     : face value
        n_sims       : number of Monte Carlo paths
        seed         : random seed for reproducibility
        """
        self.spots       = np.array(spots, dtype=float)
        self.strikes     = np.array(strikes, dtype=float)
        self.T           = T
        self.r           = r
        self.sigmas      = np.array(sigmas, dtype=float)
        self.corr        = np.array(corr_matrix, dtype=float)
        self.coupon_rate = coupon_rate
        self.notional    = notional
        self.n_sims      = n_sims
        self.seed        = seed
        self.n           = len(spots)

    def _simulate_terminal_performances(self):
        """
        Simulate S_T / K for each underlying using correlated GBM.
        Returns array of shape (n_sims, n_assets).
        """
        rng = np.random.default_rng(self.seed)

        # Cholesky decomposition to correlate the random draws
        L = np.linalg.cholesky(self.corr)

        # Independent standard normals: shape (n_sims, n_assets)
        Z = rng.standard_normal((self.n_sims, self.n))

        # Correlated normals
        Z_corr = Z @ L.T

        # GBM terminal value: S_T = S_0 * exp((r - 0.5*sigma^2)*T + sigma*sqrt(T)*Z)
        drift  = (self.r - 0.5 * self.sigmas**2) * self.T
        diffus = self.sigmas * np.sqrt(self.T) * Z_corr
        S_T = self.spots * np.exp(drift + diffus)   # shape (n_sims, n_assets)

        # Normalise to performance vs strike
        perfs = S_T / self.strikes                  # shape (n_sims, n_assets)
        return perfs

    def worst_of_put_value(self):
        """
        MC price of the worst-of put.
        Strike is 1.0 (100% of the reference level, normalised).
        Payoff per sim = notional * max(1 - worst_perf, 0)
        """
        perfs    = self._simulate_terminal_performances()
        worst    = perfs.min(axis=1)                # worst performer each path
        payoffs  = np.maximum(1.0 - worst, 0.0)    # put payoff (normalised)
        put_pv   = np.exp(-self.r * self.T) * payoffs.mean() * self.notional
        return put_pv

    def bond_pv(self):
        return self.notional * np.exp(-self.r * self.T)

    def coupon_pv(self):
        return self.notional * self.coupon_rate * self.T * np.exp(-self.r * self.T)

    def fair_value(self):
        return self.bond_pv() + self.coupon_pv() - self.worst_of_put_value()

    def fair_value_pct(self):
        return self.fair_value() / self.notional

In [ ]:
# Example: 3-stock MBRC (Novartis, Roche, ABB)

corr = np.array([
    [1.00, 0.55, 0.40],
    [0.55, 1.00, 0.45],
    [0.40, 0.45, 1.00],
])

mbrc = MBRC(
    spots        = [123.2, 318.0, 65.84],
    strikes      = [123.2, 318.0, 65.84],   # at-the-money at inception
    T            = 0.75,                    # 9 months
    r            = 0.03,
    sigmas       = [0.18, 0.22, 0.25],
    corr_matrix  = corr,
    coupon_rate  = 0.12,                    # 12% — higher because worst-of
    notional     = 100_000,
    n_sims       = 200_000
)

print(f"Bond PV           : {mbrc.bond_pv():>12,.2f}")
print(f"Coupon PV         : {mbrc.coupon_pv():>12,.2f}")
print(f"Worst-of Put (MC) : {mbrc.worst_of_put_value():>12,.2f}   ← investor is SHORT this")
print(f"──────────────────────────────────────")
print(f"Fair value        : {mbrc.fair_value():>12,.2f}")
print(f"Fair value %      : {mbrc.fair_value_pct():>12.4f}")

---
## Part 4 — Visualisations

### 4a — Payoff diagram at maturity (BRC)

In [ ]:
spots_range = np.linspace(60, 160, 300)
K = 123.0
notional = 100_000
coupon = 0.08
T = 0.5

coupon_cash = notional * coupon * T

# Payoff at maturity
payoff = np.where(
    spots_range >= K,
    notional + coupon_cash,
    notional * (spots_range / K) + coupon_cash
)

# Fair value curve (Black-Scholes)
r, sigma = 0.03, 0.20
fv = np.array([
    BRC(s, K, T, r, sigma, coupon, notional).fair_value()
    for s in spots_range
])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(spots_range, payoff / notional, label="Payoff at maturity", color="steelblue", lw=2)
ax.plot(spots_range, fv / notional, label="Fair value today (T=0.5y)", color="darkorange", lw=2, ls="--")
ax.axhline(1.0, color="grey", lw=0.8, ls=":")
ax.axvline(K, color="red", lw=1, ls=":", label=f"Strike / Barrier = {K}")
ax.set_xlabel("Spot at maturity")
ax.set_ylabel("Value as % of notional")
ax.set_title("BRC: Payoff at Maturity vs Fair Value Today")
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

### 4b — Sensitivity of fair value to vol (vega effect)

In [ ]:
vols = np.linspace(0.05, 0.50, 100)
fv_by_vol = [
    BRC(S=123, K=123, T=0.5, r=0.03, sigma=v, coupon_rate=0.08, notional=100_000).fair_value_pct()
    for v in vols
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(vols * 100, np.array(fv_by_vol) * 100, color="steelblue", lw=2)
ax.axhline(100, color="grey", lw=0.8, ls=":")
ax.set_xlabel("Implied Vol (%)")
ax.set_ylabel("Fair Value (% of notional)")
ax.set_title("Higher vol → lower RC fair value (investor is short the put)")
plt.tight_layout()
plt.show()

### 4c — Worst-of discount: single vs 2-stock vs 3-stock

In [ ]:
corr_values = np.linspace(0.0, 1.0, 15)
put_2stock = []

for rho in corr_values:
    c = np.array([[1.0, rho], [rho, 1.0]])
    m = MBRC(
        spots=[100, 100], strikes=[100, 100],
        T=0.5, r=0.03, sigmas=[0.20, 0.20],
        corr_matrix=c, coupon_rate=0.08,
        notional=1.0, n_sims=100_000
    )
    put_2stock.append(m.worst_of_put_value())

single_put = bs_put(S=100, K=100, T=0.5, r=0.03, sigma=0.20)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(corr_values, put_2stock, color="darkorange", lw=2, label="2-stock worst-of put")
ax.axhline(single_put, color="steelblue", lw=1.5, ls="--", label=f"Single-name put = {single_put:.4f}")
ax.set_xlabel("Correlation between the 2 underlyings")
ax.set_ylabel("Put Value (per unit notional)")
ax.set_title("Lower correlation → more expensive worst-of put → higher coupon possible")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

| Concept | What it means for RC investors |
|---|---|
| **Higher vol** | Lower fair value — the put you sold is worth more |
| **Lower correlation** | Worst-of put worth more — issuer can offer a higher coupon |
| **Shorter maturity** | Time value decays — theta works in your favour |
| **Spot falls** | RC value falls (negative delta) — you're short a put |
| **Spot below strike** | Capital loss at maturity proportional to the drop |

Next step: wire `BRC` and `MBRC` into the existing `ReverseConvertible` class in `src/` to replace the payoff-only model with mark-to-market pricing.